# Task A v4 — DA-only 提示词修订校准（da_v2）

只跑 **DA**，用来检验新提示词能否把 d=1 正确率从上一轮的 0.10 拉回到可用区间。题目仍是原来 `calib_set.jsonl` 里的同一批题：`d ∈ {1,2,3,5,20}` 各 100 道，共 **500** 条。不跑 CoT，不看 Task B，没有乘法。

**运行时**：选 GPU。优先 **L4**（bf16）；T4 会自动改用 fp16。依次运行全部单元。预计 L4 约 5–12 分钟（含模型下载/加载），T4 约 15–25 分钟。最后一格会下载 `raw_outputs_da_v2.jsonl` 和 `env.json`。把它们放到本地 `/workspace/new_task_a/calib_da_v2/` 后运行 `analyze_da_v2.py`。

**模型**：`Qwen/Qwen3-4B-Instruct-2507`，revision 钉在 `cdbee75f17c01a7cc42f958dc650907174af0554`。采样与上一轮相同：temperature 0.6，top_p 0.95，`max_tokens=64`。

**系统提示**（短正向指令，不再使用 “Solve silently / Do not show calculations”）：

```
Answer with only the final number, in the form #### N.
```

**Few-shot**（种子 20261112，与校准集种子 1234 不同；assistant 只有 `#### N`，不带步骤）：

示例 1（d=1，增加分支，答案 123）：

```
Grace has 112 stickers. First, if Grace has fewer than 151 stickers, she gets 11 more from a friend; otherwise, she trades away 13. Grace's class has 51 students.
How many stickers does Grace have at the end?
```

```
#### 123
```

示例 2（d=1，减少分支，答案 95）：

```
Lily has 97 beads. Lily's class has 28 students. First, Lily counts her beads. If there are more than 66, she gives 2 of them to a younger kid. If not, she finds 5 more.
How many beads does Lily have at the end?
```

```
#### 95
```


In [ ]:
!pip install -q vllm
!nvidia-smi


In [ ]:
import base64, gzip, hashlib, pathlib

FEW_SHA = "edc15cdd9c1a5a4bdb697b08390cdde9c8fae85cfe889c3f2f5ccba71a428765"
SLIM_SHA = "c3820a69e072fa6d4a6211c704dd64640af7674a85b32b2e0d06ac6b0aa57c51"

few_text = r"""{
  "prompt_version": "da_v2",
  "seed": 20261112,
  "config": "cond_addsub_dec0",
  "generator": "cond_wordproblem_v1 / generate.py",
  "note": "Two d=1 items generated with seed 20261112 (different from calib_set seed 1234). Question text, qid, number sequence, and (initial,T,a,b,ans) do not match any calib_set item. Neither example answer is a gold answer in the 500-item eval slice. The two examples have disjoint numbers, different names/items, and opposite branches. Assistant turns contain only '#### N' — no solution steps.",
  "system": "Answer with only the final number, in the form #### N.",
  "examples": [
    {
      "i": 183,
      "d": 1,
      "qid": "46e9dd1920cc5e84b96b79debdf1ba0a79037fbab9e85787ba7b7490709ba5f5",
      "q": "Grace has 112 stickers. First, if Grace has fewer than 151 stickers, she gets 11 more from a friend; otherwise, she trades away 13. Grace's class has 51 students.\nHow many stickers does Grace have at the end?",
      "ans": "123",
      "assistant": "#### 123",
      "taken": "gain",
      "name": "Grace",
      "item": "stickers",
      "initial": "112",
      "T": "151",
      "a": "13",
      "b": "11",
      "solution_not_shown_to_model": "112 < 151, so Grace gains 11: 112+11=<<112+11=123>>123\n#### 123"
    },
    {
      "i": 187,
      "d": 1,
      "qid": "7925cbbb7a9e8e71521ae260014e6b37a07393bc25759cba6dc37bec4d82a1a6",
      "q": "Lily has 97 beads. Lily's class has 28 students. First, Lily counts her beads. If there are more than 66, she gives 2 of them to a younger kid. If not, she finds 5 more.\nHow many beads does Lily have at the end?",
      "ans": "95",
      "assistant": "#### 95",
      "taken": "loss",
      "name": "Lily",
      "item": "beads",
      "initial": "97",
      "T": "66",
      "a": "2",
      "b": "5",
      "solution_not_shown_to_model": "97 > 66, so Lily loses 2: 97-2=<<97-2=95>>95\n#### 95"
    }
  ]
}
"""
pathlib.Path("few_shot.json").write_text(few_text, encoding="utf-8")

SLIM_B64 = """..."""
pathlib.Path("calib_da_v2_slim.jsonl").write_bytes(gzip.decompress(base64.b64decode(SLIM_B64)))

def sha256(p):
    return hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest()

got_few, got_slim = sha256("few_shot.json"), sha256("calib_da_v2_slim.jsonl")
assert got_few == FEW_SHA, (got_few, FEW_SHA)
assert got_slim == SLIM_SHA, (got_slim, SLIM_SHA)
n = sum(1 for _ in open("calib_da_v2_slim.jsonl", encoding="utf-8"))
print(f"wrote few_shot.json ({got_few[:12]}…) and calib_da_v2_slim.jsonl (n={n}, {got_slim[:12]}…)")
assert n == 500


In [ ]:
#!/usr/bin/env python3
"""DA-only recalibration with the v2 prompt and two d=1 few-shot examples.

Input : calib_da_v2_slim.jsonl, few_shot.json  (cwd)
Output: raw_outputs_da_v2.jsonl, env.json
Also prints per-d accuracy, parse-fail rate, and the fraction of outputs
equal to the item's initial amount.

Model : Qwen/Qwen3-4B-Instruct-2507 pinned to
        cdbee75f17c01a7cc42f958dc650907174af0554
Sampling matches calib_int / experiment_task_a.ipynb:
        temperature 0.6, top_p 0.95, DA max_tokens 64.
L4 / compute capability >= 8 uses bfloat16; T4 falls back to float16.
"""
from __future__ import annotations

import hashlib
import importlib.metadata as md
import json
import os
import re
import sys
import time
from collections import defaultdict
from decimal import Decimal, InvalidOperation

MODEL = os.environ.get("MODEL", "Qwen/Qwen3-4B-Instruct-2507")
REVISION = os.environ.get(
    "MODEL_REVISION", "cdbee75f17c01a7cc42f958dc650907174af0554"
)
IN_PATH = sys.argv[1] if len(sys.argv) > 1 else "calib_da_v2_slim.jsonl"
OUT_PATH = sys.argv[2] if len(sys.argv) > 2 else "raw_outputs_da_v2.jsonl"
FEW_PATH = os.environ.get("FEW_SHOT", "few_shot.json")

PARAMS = {"temperature": 0.6, "top_p": 0.95, "top_k_recorded_not_passed": 20}
MAX_TOKENS = 64
SYSTEM = "Answer with only the final number, in the form #### N."
PROMPT_VERSION = "da_v2"

NUM = r"-?\$?\d[\d,]*(?:\.\d+)?"


def parse(text):
    """Strict: last '#### N' line. Fallback (recorded separately): last number in text."""
    m = re.findall(r"####\s*(?:ANSWER\s*=\s*)?(" + NUM + r")", text)
    if m:
        return m[-1], "marker"
    nums = re.findall(NUM, text)
    return (nums[-1], "fallback") if nums else (None, "none")


def parse_number(s):
    if s is None:
        return None
    t = str(s).strip().replace("$", "").replace(",", "").replace(" ", "")
    try:
        return Decimal(t)
    except InvalidOperation:
        return None


def grade(pred, gold, decimals=0):
    """Same numeric grading as generate.grade (tol = 0.5 * 10^{-decimals})."""
    gp = [x for x in str(gold).split(",")] if "," in str(gold) and decimals >= 0 else [str(gold)]
    pp = [x for x in re.split(r"[;,]\s*(?=[\$\d])", str(pred).strip())] if len(gp) > 1 else [pred]
    if len(gp) != len(pp):
        return False
    tol = Decimal(5) / (Decimal(10) ** (decimals + 1))
    for p, g in zip(pp, gp):
        a, b = parse_number(p), parse_number(g)
        if a is None or b is None or abs(a - b) > tol:
            return False
    return True


def seed_of(qid, mode="DA"):
    return int(hashlib.sha256(f"{qid}|{mode}".encode()).hexdigest()[:8], 16)


def build_messages(question, few_shot):
    msgs = [{"role": "system", "content": few_shot.get("system", SYSTEM)}]
    for ex in few_shot["examples"]:
        msgs.append({"role": "user", "content": ex["q"]})
        msgs.append({"role": "assistant", "content": ex["assistant"]})
    msgs.append({"role": "user", "content": question})
    return msgs


def summarize(rows):
    by = defaultdict(list)
    for r in rows:
        by[r["d"]].append(r)
    print("\n=== DA v2 inline metrics ===")
    print(f"{'d':>3} {'n':>4} {'correct':>8} {'acc':>7} {'parse_fail':>11} "
          f"{'fallback':>9} {'echo_init':>10} {'trunc':>7} {'mean_tok':>9}")
    for d in sorted(by):
        g = by[d]
        n = len(g)
        k = sum(r["correct"] for r in g)
        pf = sum(r["parse_how"] == "none" for r in g)
        fb = sum(r["parse_how"] == "fallback" for r in g)
        echo = sum(r["echo_initial"] for r in g)
        tr = sum(r["trunc"] for r in g)
        mt = sum(r["n_out_tokens"] for r in g) / n if n else 0
        print(f"{d:>3} {n:>4} {k:>8} {k/n if n else 0:>7.3f} {pf/n if n else 0:>11.3f} "
              f"{fb/n if n else 0:>9.3f} {echo/n if n else 0:>10.3f} {tr/n if n else 0:>7.3f} {mt:>9.1f}")
    n = len(rows)
    k = sum(r["correct"] for r in rows)
    echo = sum(r["echo_initial"] for r in rows)
    pf = sum(r["parse_how"] == "none" for r in rows)
    print(f"ALL {n:>4} {k:>8} {k/n if n else 0:>7.3f} {pf/n if n else 0:>11.3f} "
          f"{'':>9} {echo/n if n else 0:>10.3f}")
    print("parse_fail = no number parsed; echo_init = parsed value equals the initial amount.")


def main():
    import torch
    from vllm import LLM, SamplingParams

    rows = [json.loads(l) for l in open(IN_PATH, encoding="utf-8")]
    few_shot = json.load(open(FEW_PATH, encoding="utf-8"))
    sys_prompt = few_shot.get("system", SYSTEM)
    if sys_prompt != SYSTEM:
        print("WARNING: few_shot.json system differs from built-in SYSTEM", file=sys.stderr)
    for ex in few_shot["examples"]:
        if ex["assistant"] != f"#### {ex['ans']}":
            raise ValueError(f"few-shot assistant must be exactly '#### N', got {ex['assistant']!r}")

    cap = torch.cuda.get_device_capability(0)
    dtype = "bfloat16" if cap[0] >= 8 else "float16"
    print(f"GPU={torch.cuda.get_device_name(0)} cc={list(cap)} dtype={dtype}")
    print(f"model={MODEL} revision={REVISION} n={len(rows)} max_tokens={MAX_TOKENS}")
    print(f"system={sys_prompt!r}")

    llm = LLM(
        model=MODEL,
        revision=REVISION,
        tokenizer_revision=REVISION,
        dtype=dtype,
        max_model_len=4096,
        gpu_memory_utilization=0.88,
        seed=0,
    )
    tok = llm.get_tokenizer()
    messages = [build_messages(r["q"], few_shot) for r in rows]
    prompts = [
        tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
        for m in messages
    ]
    sps = [
        SamplingParams(
            temperature=PARAMS["temperature"],
            top_p=PARAMS["top_p"],
            max_tokens=MAX_TOKENS,
            seed=seed_of(r["qid"], "DA"),
        )
        for r in rows
    ]
    t0 = time.time()
    outs = llm.generate(prompts, sps)
    el = time.time() - t0

    scored = []
    with open(OUT_PATH, "w", encoding="utf-8") as f:
        for r, o in zip(rows, outs):
            c = o.outputs[0]
            pred, how = parse(c.text)
            item = {
                "qid": r["qid"],
                "d": r["d"],
                "i": r["i"],
                "mode": "DA",
                "gold": r["ans"],
                "answer_decimals": r["answer_decimals"],
                "initial": r["initial"],
                "text": c.text,
                "finish_reason": c.finish_reason,
                "n_out_tokens": len(c.token_ids),
                "n_prompt_tokens": len(o.prompt_token_ids),
                "seed": seed_of(r["qid"], "DA"),
                "prompt_version": PROMPT_VERSION,
            }
            item["pred"] = pred
            item["parse_how"] = how
            item["correct"] = bool(pred is not None and grade(pred, r["ans"], r["answer_decimals"]))
            item["trunc"] = c.finish_reason == "length"
            item["echo_initial"] = bool(
                pred is not None and grade(pred, r["initial"], r["answer_decimals"])
            )
            # raw file keeps the same core fields as calib_int, plus initial / prompt_version
            raw = {k: item[k] for k in (
                "qid", "d", "i", "mode", "gold", "answer_decimals", "initial",
                "text", "finish_reason", "n_out_tokens", "n_prompt_tokens",
                "seed", "prompt_version",
            )}
            f.write(json.dumps(raw, ensure_ascii=False) + "\n")
            scored.append(item)

    try:
        from huggingface_hub import model_info
        rev = model_info(MODEL, revision=REVISION).sha
    except Exception as e:
        rev = f"unresolved: {e}"

    env = {
        "prompt_version": PROMPT_VERSION,
        "model": MODEL,
        "model_revision": rev,
        "requested_model_revision": REVISION,
        "dtype": dtype,
        "gpu": torch.cuda.get_device_name(0),
        "compute_capability": list(cap),
        "loading": {
            "dtype": dtype,
            "gpu_memory_utilization": 0.88,
            "max_model_len": 4096,
            "seed": 0,
            "revision": REVISION,
            "tokenizer_revision": REVISION,
        },
        "vllm": md.version("vllm"),
        "torch": torch.__version__,
        "transformers": md.version("transformers"),
        "sampling": PARAMS,
        "max_tokens": {"DA": MAX_TOKENS},
        "system_prompts": {"DA": sys_prompt},
        "few_shot": {
            "path": FEW_PATH,
            "seed": few_shot.get("seed"),
            "n_examples": len(few_shot["examples"]),
            "qids": [ex["qid"] for ex in few_shot["examples"]],
            "assistants": [ex["assistant"] for ex in few_shot["examples"]],
        },
        "depths": sorted({r["d"] for r in rows}),
        "n_jobs": len(rows),
        "generate_seconds": round(el, 1),
        "input": IN_PATH,
        "output": OUT_PATH,
    }
    json.dump(env, open("env.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
    print(json.dumps(env, indent=2, ensure_ascii=False))
    summarize(scored)


if __name__ == "__main__":
    main()


In [ ]:
from google.colab import files
files.download("raw_outputs_da_v2.jsonl")
files.download("env.json")
